In [69]:
import os
import pandas as pd
import pyspark
from pyspark.sql.functions import date_format, to_date, datediff, col, unix_timestamp, max
from pyspark.sql import SparkSession
from pyspark.sql import types

In [6]:
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.ui.port", "4040") \
    .appName('test') \
    .getOrCreate()

In [7]:
# Check original schema
df_yellow = spark.read \
    .option("header", "true") \
        .parquet('download_scripts/data/yellow_tripdata_2025-11.parquet')

In [8]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [9]:
year = 2025
month = 11
output_path = f'download_scripts/data/pq/yellow/{year}/{month:02d}/'

df_yellow \
    .repartition(4) \
    .write.parquet(output_path)

In [22]:
# Reading file sizes
path = "download_scripts/data/pq/yellow/2025/11"

def list_files_with_size(path):
    for entry in os.scandir(path):
        if entry.is_file() and entry.name.lower().endswith(".parquet"):
            size = entry.stat().st_size
            unit = 'bytes'
            if size > 1024:
                size /= 1024
                unit = 'KB'
            if size > 1024:
                size /= 1024
                unit = 'MB'
            if size > 1024:
                size /= 1024
                unit = 'GB'
            print(f"{entry.name} → {size:.2f} {unit}")


In [23]:
size = list_files_with_size(path)
print(f"Directory size: {size} bytes")

part-00001-8c8d6ad5-b246-420e-9426-45c6b4bc6437-c000.snappy.parquet → 24.41 MB
part-00000-8c8d6ad5-b246-420e-9426-45c6b4bc6437-c000.snappy.parquet → 24.41 MB
part-00002-8c8d6ad5-b246-420e-9426-45c6b4bc6437-c000.snappy.parquet → 24.42 MB
part-00003-8c8d6ad5-b246-420e-9426-45c6b4bc6437-c000.snappy.parquet → 24.42 MB
Directory size: None bytes


In [26]:
df_yellow.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [33]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [37]:
df_yellow.filter(to_date(df_yellow.tpep_pickup_datetime) == '2025-11-15').count()

162604

In [71]:
df_with_days = df_yellow.withColumn(
    "trip_time_hours", (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600)

In [72]:
df_with_days.select("tpep_dropoff_datetime", "tpep_pickup_datetime", "trip_time_hours").show()

+---------------------+--------------------+--------------------+
|tpep_dropoff_datetime|tpep_pickup_datetime|     trip_time_hours|
+---------------------+--------------------+--------------------+
|  2025-11-01 00:13:25| 2025-11-01 00:13:25|                 0.0|
|  2025-11-01 01:01:22| 2025-11-01 00:49:07| 0.20416666666666666|
|  2025-11-01 00:20:41| 2025-11-01 00:07:19| 0.22277777777777777|
|  2025-11-01 01:01:03| 2025-11-01 00:00:00|              1.0175|
|  2025-11-01 00:49:32| 2025-11-01 00:18:50|  0.5116666666666667|
|  2025-11-01 00:31:39| 2025-11-01 00:21:11| 0.17444444444444446|
|  2025-11-01 00:25:44| 2025-11-01 00:07:31|  0.3036111111111111|
|  2025-11-01 01:38:55| 2025-11-01 00:46:52|              0.8675|
|  2025-11-01 01:02:05| 2025-11-01 00:56:59|               0.085|
|  2025-11-01 00:39:25| 2025-11-01 00:10:43| 0.47833333333333333|
|  2025-11-01 00:42:25| 2025-11-01 00:00:03|  0.7061111111111111|
|  2025-11-01 00:56:46| 2025-11-01 00:43:53| 0.21472222222222223|
|  2025-11

In [73]:
df_with_days.agg(max("trip_time_hours").alias("max_trip_time_hours")).show()

+-------------------+
|max_trip_time_hours|
+-------------------+
|  90.64666666666666|
+-------------------+



In [77]:
# Importing zone data
df_zones = spark.read.parquet('download_scripts/data/zones/')

In [78]:
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [80]:
df_yellow.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [81]:
# join zone data with yellow trip data
df_result = df_yellow.join(df_zones, df_yellow.PULocationID == df_zones.LocationID)

In [84]:
df_result.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+-------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|               Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+--

In [87]:
df_pickup_report = df_result.select("tpep_pickup_datetime", "PULocationID", "LocationID", "Borough", "Zone", "service_zone")

In [91]:
df_pickup_report.groupBy("Zone").count().show()

+--------------------+------+
|                Zone| count|
+--------------------+------+
|           Homecrest|   745|
|Governor's Island...|     1|
|              Corona|  1041|
|    Bensonhurst West|   750|
|      Newark Airport|   593|
|          Douglaston|   134|
|East Concourse/Co...|  1713|
|          Mount Hope|  1139|
|      Pelham Parkway|   560|
|         Marble Hill|   229|
|           Rego Park|  1014|
|Upper East Side S...|194593|
|       Dyker Heights|   455|
|Heartland Village...|    22|
|   Kew Gardens Hills|   543|
|       Rikers Island|     4|
|     Jackson Heights|  4695|
|             Bayside|   330|
|TriBeCa/Civic Center| 55233|
|      Yorkville West| 76718|
+--------------------+------+
only showing top 20 rows


In [ ]:
spark.stop()